# Credit Card Fraud Detection - Google Colab

Use this notebook if you want to run the project in Google Colab. Upload `archive (2).zip`, install the dependencies, extract the dataset, and train the models.

## 1. Upload Dataset

Run this cell and upload your Kaggle dataset zip file. The zip should contain `fraudTrain.csv` and `fraudTest.csv`.

In [ ]:
from google.colab import files
uploaded = files.upload()
zip_path = next(iter(uploaded.keys()))
zip_path

## 2. Install Dependencies

In [ ]:
!pip install pandas numpy scikit-learn joblib streamlit -q

## 3. Extract Dataset

In [ ]:
import zipfile
from pathlib import Path

Path('data').mkdir(exist_ok=True)
with zipfile.ZipFile(zip_path) as z:
    for name in z.namelist():
        if name.endswith('fraudTrain.csv') or name.endswith('fraudTest.csv'):
            z.extract(name, 'data')

# Move files to data/ if the zip extracted inside a folder
for file in Path('data').rglob('fraud*.csv'):
    target = Path('data') / file.name
    if file != target:
        target.write_bytes(file.read_bytes())

list(Path('data').glob('*.csv'))

## 4. Train and Compare Models

This cell trains Logistic Regression, Decision Tree, and Random Forest on a sample of the data for faster Colab execution. Change `sample_frac` to `1.0` for full training.

In [ ]:
import joblib
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, classification_report, roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier

sample_frac = 0.2
train_df = pd.read_csv('data/fraudTrain.csv').sample(frac=sample_frac, random_state=42)
test_df = pd.read_csv('data/fraudTest.csv').sample(frac=sample_frac, random_state=42)

def add_features(df):
    df = df.copy()
    transaction_time = pd.to_datetime(df['trans_date_trans_time'], errors='coerce')
    birth_date = pd.to_datetime(df['dob'], errors='coerce')
    df['hour'] = transaction_time.dt.hour
    df['day_of_week'] = transaction_time.dt.dayofweek
    df['month'] = transaction_time.dt.month
    df['age'] = ((transaction_time - birth_date).dt.days / 365.25).round(1)
    return df

numeric_features = ['amt', 'lat', 'long', 'city_pop', 'unix_time', 'merch_lat', 'merch_long', 'age', 'hour', 'day_of_week', 'month']
categorical_features = ['category', 'gender', 'state']
drop_columns = ['Unnamed: 0', 'cc_num', 'first', 'last', 'street', 'city', 'job', 'merchant', 'trans_num', 'trans_date_trans_time', 'dob']

def split_xy(df):
    df = add_features(df).drop(columns=[c for c in drop_columns if c in df.columns])
    return df.drop(columns=['is_fraud']), df['is_fraud']

x_train, y_train = split_xy(train_df)
x_test, y_test = split_xy(test_df)

def preprocessor():
    return ColumnTransformer([
        ('num', Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())]), numeric_features),
        ('cat', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('onehot', OneHotEncoder(handle_unknown='ignore'))]), categorical_features),
    ])

models = {
    'logistic_regression': LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42),
    'decision_tree': DecisionTreeClassifier(max_depth=12, min_samples_leaf=20, class_weight='balanced', random_state=42),
    'random_forest': RandomForestClassifier(n_estimators=120, max_depth=16, min_samples_leaf=10, class_weight='balanced_subsample', n_jobs=-1, random_state=42),
}

results = []
trained = {}
for name, clf in models.items():
    pipe = Pipeline([('preprocessor', preprocessor()), ('model', clf)])
    pipe.fit(x_train, y_train)
    pred = pipe.predict(x_test)
    scores = pipe.predict_proba(x_test)[:, 1]
    results.append({
        'model': name,
        'roc_auc': roc_auc_score(y_test, scores),
        'average_precision': average_precision_score(y_test, scores),
    })
    trained[name] = pipe

results_df = pd.DataFrame(results).sort_values('average_precision', ascending=False)
results_df

## 5. Save Best Model

In [ ]:
Path('models').mkdir(exist_ok=True)
best_model_name = results_df.iloc[0]['model']
joblib.dump({'model_name': best_model_name, 'model': trained[best_model_name]}, 'models/best_fraud_model.joblib')
print('Best model:', best_model_name)